In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 5)

In [0]:
path = '/Volumes/spark_definitive_guide/spark_book/activity-data/'
static = spark.read.json(path)

In [0]:
streaming = spark\
    .readStream\
    .schema(static.schema)\
    .option("maxFilesPerTrigger", 10)\
    .json(path)

# Windows on Event Time

Converter o campo Creation_Time de long para timestamp

In [0]:
withEventTime = streaming\
    .selectExpr(
    "*",
    "cast(cast(Creation_Time as double)/1000000000 as timestamp) as event_time")

In [0]:
from pyspark.sql.functions import window, col 
withEventTime.groupBy(window(col("event_time"), "10 minutes")).count()\
    .writeStream\
    .queryName("pyevents_per_window")\
    .format("memory")\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/checkpoints/pyevents_per_window")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()


In [0]:
spark.sql("SELECT * FROM  pyevents_per_window").printSchema()

In [0]:
%sql
SELECT * FROM  pyevents_per_window

# Tumbling windows

In [0]:
from pyspark.sql.functions import window, col

withEventTime.groupBy(window(col("event_time"), "10 minutes"), "User").count()\
    .writeStream\
    .queryName("user_event_per_window")\
    .format("memory")\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/checkpoints/user_event_per_window")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()


In [0]:
%sql
SELECT * FROM  user_event_per_window

# Sliding windows

In [0]:
from pyspark.sql.functions import window , col

withEventTime.groupBy(window(col("event_time"), "10 minutes", "5 minutes"))\
    .count()\
    .writeStream\
    .queryName("pyevents_per_window")\
    .format("memory")\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/checkpoints/pyevents_per_window")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()

In [0]:
%sql
select * from pyevents_per_window

# Watermark

In [0]:
from pyspark.sql.functions import window, col

withEventTime\
    .withWatermark("event_time", "30 minutes")\
    .groupBy(window(col("event_time"), "10 minutes", "5 minutes"))\
    .count()\
    .writeStream\
    .queryName("pyevents_per_window")\
    .format("memory")\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/checkpoints/pyevents_per_window")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()

In [0]:
%sql
select * from pyevents_per_window

# Dropping duplicates

In [0]:
# Drop duplicates from stream
withEventTime\
    .withWatermark("event_time", "5 seconds")\
    .dropDuplicates(["User", "event_time"])\
    .groupBy("User")\
    .count()\
    .writeStream\
    .queryName("pydeduplicated")\
    .option("checkpointLocation", "/Volumes/spark_definitive_guide/spark_book/activity-data/checkpoints/pydeduplicated")\
    .format("memory")\
    .outputMode("complete")\
    .trigger(availableNow=True)\
    .start()

In [0]:
%sql
select * from pydeduplicated